# Model Selection — Content Generation

Runs the `model-selection` "Content Generation" scenario end-to-end from a notebook:

1. **Requirements** — verify `aiperf` is installed and the scenario script/prompts exist.
2. **Test run** — invoke `model-selection/scripts/run_content_generation.sh` (the source of truth for this scenario, per `CLAUDE.md`) rather than reimplementing the `aiperf profile` call here.
3. **Results analysis** — load AIPerf's export (`profile_export_aiperf.csv` / `.json`) and summarize the metrics documented in `docs/metrics/model-selection.md`.

See `docs/scenarios/model-selection.md` for the scenario definition (ISL/OSL/turns/concurrency) and `docs/metrics/model-selection.md` for what each metric means.

This notebook does not alter the scenario script — it drives it, so the script remains the single source of truth per scenario (script-as-config).

## 1. Requirements

- `aiperf` CLI installed and on `PATH` (from the NGC AIPerf image, or `pip install aiperf` in a jumphost environment).
- Reachable OpenAI-compatible endpoint (NIM, vLLM, TGI, ...).
- This notebook's own Python deps: `pip install -r notebooks/requirements.txt` (pandas, matplotlib — for step 3 only).

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "model-selection").exists() else Path.cwd()
assert (REPO_ROOT / "model-selection").exists(), (
    f"Could not find model-selection/ from {Path.cwd()} — run this notebook from the notebooks/ directory."
)

SCRIPT_PATH = REPO_ROOT / "model-selection" / "scripts" / "run_content_generation.sh"
PROMPTS_PATH = REPO_ROOT / "model-selection" / "prompts" / "content_generation.jsonl"

print(f"Repo root:     {REPO_ROOT}")
print(f"Script:        {SCRIPT_PATH} (exists: {SCRIPT_PATH.exists()})")
print(f"Prompts file:  {PROMPTS_PATH} (exists: {PROMPTS_PATH.exists()})")

In [ ]:
aiperf_path = shutil.which("aiperf")
if aiperf_path is None:
    print("aiperf CLI not found on PATH.")
    print("Install it (e.g. from the NGC AIPerf image, or `pip install aiperf`) before running Section 2.")
else:
    print(f"aiperf found at: {aiperf_path}")
    version = subprocess.run(["aiperf", "--version"], capture_output=True, text=True)
    print((version.stdout or version.stderr).strip())

## 2. Test run

Configure the target model/endpoint below, then run the scenario script as a subprocess. All parameters (env vars) map 1:1 to what `run_content_generation.sh` accepts — see the script header for the full list (tokenizer, HF token/cache, output dir, etc.).

Concurrency defaults to the V1 baseline (`1`); set it to `5`, `10`, or `25` to run a sweep point instead (see `docs/scenarios/model-selection.md`).

In [ ]:
# ---- Required ----------------------------------------------------------
MODEL = ""          # e.g. "granite-3-1-8b-instruct"
URL = ""             # e.g. "http://localhost:8000"

# ---- Optional (leave empty to use the script's defaults / skip) --------
TOKENIZER_PATH = ""  # local dir or HF repo id; empty = use MODEL as tokenizer name
HF_TOKEN = ""         # only needed for gated/private HF tokenizer repos
HF_HOME = ""          # override HF cache dir if the default isn't writable
CONCURRENCY = "1"     # baseline; sweep: 1/5/10/25
OUTPUT_DIR = str(REPO_ROOT / "model-selection" / "artifacts" / "content-generation")

assert MODEL, "Set MODEL before running."
assert URL, "Set URL before running."

In [ ]:
env = os.environ.copy()
env["MODEL"] = MODEL
env["URL"] = URL
env["TOKENIZER_PATH"] = TOKENIZER_PATH
env["HF_TOKEN"] = HF_TOKEN
if HF_HOME:
    env["HF_HOME"] = HF_HOME
env["CONCURRENCY"] = CONCURRENCY
env["OUTPUT_DIR"] = OUTPUT_DIR

# All env vars the script reads via `read -p` must be set (even to "") so it
# never blocks on interactive input, since there's no TTY from a notebook cell.

result = subprocess.run(
    ["bash", str(SCRIPT_PATH)],
    env=env,
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"run_content_generation.sh exited with code {result.returncode}")

## 3. Results analysis

AIPerf writes its export to `OUTPUT_DIR`. Per AIPerf's export docs, the artifact directory contains:

- `profile_export_aiperf.csv` — aggregated statistics (one metric per row: avg/min/max/p50/p90/p99/std, spreadsheet-friendly).
- `profile_export_aiperf.json` — the same aggregated statistics plus the run's configuration, as one JSON object.
- `profile_export.jsonl` — per-request raw records (one JSON object per line), for anything not covered by the aggregates.
- `inputs.json` — the exact input payloads sent.

This is raw AIPerf output — per `CLAUDE.md`, there is deliberately no reformatted report layer, so this notebook reads the export as-is rather than transforming it into a different schema.

In [ ]:
import pandas as pd

artifact_dir = Path(OUTPUT_DIR)
csv_path = artifact_dir / "profile_export_aiperf.csv"
json_path = artifact_dir / "profile_export_aiperf.json"

assert csv_path.exists(), f"Expected export not found: {csv_path}. Did the run in Section 2 complete?"

stats = pd.read_csv(csv_path)
pd.set_option("display.max_rows", None)
stats

In [ ]:
with open(json_path) as f:
    export = json.load(f)

config_keys = [k for k in export.keys() if k not in stats.iloc[:, 0].tolist()]
print("Top-level keys in profile_export_aiperf.json (beyond per-metric stats):")
for k in config_keys:
    print(f"  - {k}")

### Key metrics (per `docs/metrics/model-selection.md`)

Content Generation is streaming, single-turn, long-form output (OSL 800). The metrics that matter most for a model-selection decision on this scenario:

- **TTFT** (`time_to_first_token`) — responsiveness before generation starts.
- **ITL** (`inter_token_latency`) — streaming smoothness during the long generation.
- **Output Token Throughput per User** (`output_token_throughput_per_user`) — perceived generation speed.
- **Output Sequence Length** (`output_sequence_length`) — sanity check against the 800-token target; unexpectedly short outputs flag truncation.
- **Goodput** (`goodput`) — the most decision-relevant single number; % of requests meeting TTFT/ITL/latency SLOs together.

The cell below filters the exported stats table to these rows if present (row-matching is tolerant of AIPerf's exact display-name formatting, e.g. "Time To First Token (ms)").

In [ ]:
metric_col = stats.columns[0]

key_metric_patterns = [
    "time to first token",
    "inter token latency",
    "output token throughput per user",
    "output sequence length",
    "goodput",
    "request latency",
    "error",
]

mask = stats[metric_col].str.lower().apply(
    lambda name: any(pattern in name for pattern in key_metric_patterns)
)
key_metrics = stats[mask]
key_metrics

In [ ]:
import matplotlib.pyplot as plt

# Plot avg vs. p99 for the latency-shaped metrics, if those columns are present.
latency_rows = stats[stats[metric_col].str.lower().str.contains("token|latency", regex=True)]
available_stat_cols = [c for c in ["avg", "p50", "p90", "p99"] if c in stats.columns]

if not latency_rows.empty and available_stat_cols:
    ax = latency_rows.set_index(metric_col)[available_stat_cols].plot(
        kind="barh", figsize=(9, max(3, 0.4 * len(latency_rows)))
    )
    ax.set_xlabel("value")
    ax.set_title("Content Generation — latency/token metrics")
    plt.tight_layout()
    plt.show()
else:
    print("Expected stat columns (avg/p50/p90/p99) not found in this export — inspect `stats.columns` above.")

### Notes

- If you ran a concurrency sweep (rerunning Section 2 with `CONCURRENCY` set to `5`, `10`, `25`), set a distinct `OUTPUT_DIR` per run (e.g. suffix with the concurrency value) so exports don't overwrite each other, then load and concatenate each `profile_export_aiperf.csv` here to compare across concurrency.
- To compare across **models**, rerun Section 2 with a different `MODEL`/`URL`/`TOKENIZER_PATH` and a distinct `OUTPUT_DIR`, then repeat this section per model's export.